# 01 — Process
Per deployment: annotation QC flags, instrument processing
(`suna_datalogger` / `suna_instrument`), burst-averaging, met-derived T/S
substitution (buoy instruments only), the Plant et al. (2023) T-S(-P)
correction, stream merging, and drift correction against the post-cruise
calibration.

In [ ]:
import sys, os
import yaml
import numpy as np
import xarray as xr
sys.path.insert(0, '..')

from ooi_data_explorations.common import get_annotations, load_kdata, add_annotation_qc_flags, list_deployments
from ooi_data_explorations.combine_data import combine_datasets
from ooi_data_explorations.uncabled.process_nutnr import suna_datalogger, suna_instrument
from ooi_data_explorations.uncabled.process_metbk import metct_instrument

from nitrate import resample, calibration

In [ ]:
config = yaml.safe_load(open('../config/GI01SUMO-SBD11-08-NUTNRB000.yaml'))
refdes = config['refdes']
site, node, sensor = refdes.split('-', 2)
data_dir = f"../{config['paths']['data_dir']}"

## 1. Helper: load co-located met data (buoy instruments only)

In [ ]:
def load_met_data(dep):
    """Returns burst-averaged METBK data for buoy (SBD) NUTNR deployments,
    or None for instruments with a co-located CTD."""
    if not config.get('met_refdes'):
        return None
    met_refdes = config['met_refdes']
    met_site, met_node, met_sensor = met_refdes.split('-', 2)
    met_stream = config['met_streams']['recovered_inst']
    met_data = load_kdata(met_site, met_node, met_sensor, 'recovered_inst', met_stream,
                           tag=f'deployment{dep}_{met_refdes}*.nc')
    met_data = metct_instrument(met_data, burst=False)
    return resample.met_burst_resample(met_data)

## 2. Helper: process one delivery method's stream

In [ ]:
def process_stream(dep, method, processor, met_data):
    stream = config['streams'][method]
    annotations = get_annotations(site, node, sensor)
    data = load_kdata(site, node, sensor, method, stream, tag=f'deployment{dep}_{refdes}*.nc')
    data = add_annotation_qc_flags(data, annotations)

    # recovered_host serial_number sometimes carries an extra string4 dim
    if method == 'recovered_host' and 'string4' in data['serial_number'].dims:
        data['serial_number'] = data['serial_number'].isel(string4=0)

    data = processor(data, burst=False)
    data = resample.burst_resample(data)

    if met_data is not None:
        met_interp = met_data.interp_like(data)
        data['sea_water_practical_salinity'] = met_interp['sea_surface_salinity']
        data['sea_water_temperature'] = met_interp['sea_surface_temperature']

    return calibration.add_plant_correction(data, site, node, sensor)

## 3. Try it on one deployment

In [ ]:
dN = 9
dep = str(dN).zfill(4)

met_data = load_met_data(dep)
tdata = process_stream(dep, 'telemetered', suna_datalogger, met_data)
hdata = process_stream(dep, 'recovered_host', suna_datalogger, met_data)
idata = process_stream(dep, 'recovered_inst', suna_instrument, met_data)

tdata = tdata.drop_vars('internal_timestamp', errors='ignore')
hdata = hdata.drop_vars('internal_timestamp', errors='ignore')
idata = idata.drop_vars('internal_timestamp', errors='ignore')
data = combine_datasets(tdata, hdata, idata, None)
data = calibration.plant_drift_correction(data, site, node, sensor)
data

## 4. Run for every deployment and save

In [ ]:
all_deployments = sorted(list_deployments(site, node, sensor))
drop = set(config.get('deployments_to_drop', []))
deployments = [d for d in all_deployments if d not in drop]

for dN in deployments:
    dep = str(dN).zfill(4)
    met_data = load_met_data(dep)

    tdata = process_stream(dep, 'telemetered', suna_datalogger, met_data)
    hdata = process_stream(dep, 'recovered_host', suna_datalogger, met_data)
    idata = process_stream(dep, 'recovered_inst', suna_instrument, met_data)

    tdata = tdata.drop_vars('internal_timestamp', errors='ignore')
    hdata = hdata.drop_vars('internal_timestamp', errors='ignore')
    idata = idata.drop_vars('internal_timestamp', errors='ignore')
    dep_data = combine_datasets(tdata, hdata, idata, None)
    dep_data = calibration.plant_drift_correction(dep_data, site, node, sensor)

    outpath = f'{data_dir}{refdes}_deployment{dep}_drift_corrected.nc'
    dep_data.to_netcdf(outpath, format='netcdf4', engine='h5netcdf')
    print(f'Deployment {dep}: saved -> {outpath}')